# Data splits

## 1. Import libraries

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

## 2. Load data

In [2]:
df = pd.read_csv("..\\dataset\\fragments_metadata.csv")
df

,id,name,age,gender,position,record_id,segment,label,category,duration
0,1,P1,4.3,1,p4,7545,0,Normal,Normal,1.57725
1,2,P1,4.3,1,p4,7545,1,Rhonchi,Adventitious,0.95725
2,3,P1,4.3,1,p4,7545,2,Normal,Normal,1.01225
3,4,P2,5.3,0,p1,25271,0,Normal,Normal,2.12525
4,5,P2,5.3,0,p4,25284,0,Fine Crackle,Adventitious,1.93425
...,...,...,...,...,...,...,...,...,...,...
24573,24574,P957,8.4,0,p8,32670,3,Normal,Normal,0.94025
24574,24575,P957,8.4,0,p8,32670,4,Normal,Normal,0.99525
24575,24576,P957,8.4,0,p8,32670,5,Normal,Normal,0.68525
24576,24577,P957,8.4,0,p8,32670,6,Normal,Normal,1.19925


## 3. Train-test split

### 3.1. Train/Validation-Test sets

In [3]:
X = df["id"].values
y = df["label"].values
groups = df["name"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Split
train_val_idx, test_idx = next(sgkf.split(X, y, groups))

df_train_val = df.iloc[train_val_idx].copy()
df_test = df.iloc[test_idx].copy()

In [4]:
print(f"Train/Val: {len(df_train_val)} ({len(df_train_val) / len(df) * 100:.2f}%)")
print(f"Test: {len(df_test)} ({len(df_test) / len(df) * 100:.2f}%)")

print("\nDistribución train/val:")
print(df_train_val["label"].value_counts(normalize=True))

print("\nDistribución test:")
print(df_test["label"].value_counts(normalize=True))

Train/Val: 19663 (80.00%)
Test: 4915 (20.00%)

Distribución train/val:
label
Normal            0.763770
Fine Crackle      0.143620
Wheeze            0.061181
Wheeze+Crackle    0.012409
Rhonchi           0.008900
Coarse Crackle    0.007222
Stridor           0.002899
Name: proportion, dtype: float64

Distribución test:
label
Normal            0.763784
Fine Crackle      0.143642
Wheeze            0.061445
Wheeze+Crackle    0.012004
Rhonchi           0.008545
Coarse Crackle    0.007121
Stridor           0.003459
Name: proportion, dtype: float64


### 3.2. Train-Validation sets

In [5]:
X_tv = df_train_val["id"].values
y_tv = df_train_val["label"].values
groups_tv = df_train_val["name"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

df_train_val["fold"] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(X_tv, y_tv, groups_tv), start=1):
    df_train_val.iloc[val_idx, df_train_val.columns.get_loc("fold")] = fold

In [6]:
print("Number of samples per fold:")
print(df_train_val["fold"].value_counts().sort_index())

for fold in sorted(df_train_val["fold"].unique()):
    val = df_train_val[df_train_val["fold"] == fold]
    train = df_train_val[df_train_val["fold"] != fold]

    print(f"\nFold {fold}")
    print("Train samples:", len(train), f"({len(train)/len(df_train_val)*100:.2f}%)", "Val samples:", len(val), f"({len(val)/len(df_train_val)*100:.2f}%)")
    print("Train patients:", train["name"].nunique(), "Val patients:", val["name"].nunique())
    print("Train label distribution:")
    print(train["label"].value_counts(normalize=True))
    print("Val label distribution:")
    print(val["label"].value_counts(normalize=True))

Number of samples per fold:
fold
1    3929
2    3933
3    3932
4    3936
5    3933
Name: count, dtype: int64

Fold 1
Train samples: 15734 (80.02%) Val samples: 3929 (19.98%)
Train patients: 618 Val patients: 154
Train label distribution:
label
Normal            0.763569
Fine Crackle      0.143638
Wheeze            0.061141
Wheeze+Crackle    0.012330
Rhonchi           0.008898
Coarse Crackle    0.007373
Stridor           0.003051
Name: proportion, dtype: float64
Val label distribution:
label
Normal            0.764571
Fine Crackle      0.143548
Wheeze            0.061339
Wheeze+Crackle    0.012726
Rhonchi           0.008908
Coarse Crackle    0.006617
Stridor           0.002291
Name: proportion, dtype: float64

Fold 2
Train samples: 15730 (80.00%) Val samples: 3933 (20.00%)
Train patients: 614 Val patients: 158
Train label distribution:
label
Normal            0.763827
Fine Crackle      0.143611
Wheeze            0.061157
Wheeze+Crackle    0.012397
Rhonchi           0.008837
Coarse Crack

## 4. Save splits

In [7]:
df_test["fold"] = -1
new_df = pd.concat([df_train_val, df_test]).sort_values("id")
new_df

,id,name,age,gender,position,record_id,segment,label,category,duration,fold
0,1,P1,4.3,1,p4,7545,0,Normal,Normal,1.57725,-1
1,2,P1,4.3,1,p4,7545,1,Rhonchi,Adventitious,0.95725,-1
2,3,P1,4.3,1,p4,7545,2,Normal,Normal,1.01225,-1
3,4,P2,5.3,0,p1,25271,0,Normal,Normal,2.12525,5
4,5,P2,5.3,0,p4,25284,0,Fine Crackle,Adventitious,1.93425,5
...,...,...,...,...,...,...,...,...,...,...,...
24573,24574,P957,8.4,0,p8,32670,3,Normal,Normal,0.94025,-1
24574,24575,P957,8.4,0,p8,32670,4,Normal,Normal,0.99525,-1
24575,24576,P957,8.4,0,p8,32670,5,Normal,Normal,0.68525,-1
24576,24577,P957,8.4,0,p8,32670,6,Normal,Normal,1.19925,-1


In [8]:
new_df.to_csv("splits/splits.csv", index=False)